# 🎯 DPO Alignment — Where Is My Paisa Finance Model

**Optional Stage B** from `research/finetune.md`.

Run DPO (Direct Preference Optimization) on top of the SFT-trained model to improve:
- Response tone consistency
- Brevity and actionability
- Safety boundary behavior

**Requirements**: Run a SFT notebook first to get a base SFT checkpoint.

> **GPU**: T4 (16 GB) for 1B models, A100 for 3B+

In [ ]:
# @title Install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]", "trl>=0.8.6", "peft", "accelerate",
    "bitsandbytes", "datasets", "huggingface_hub", "--quiet", "--upgrade"], check=True)
print("✅ Installation complete")

In [ ]:
# @title Hardware check
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU required. Runtime → Change runtime type → T4/A100")
print(f"GPU: {torch.cuda.get_device_name(0)}")
from unsloth import is_bfloat16_supported
print(f"BF16: {is_bfloat16_supported()}")

In [ ]:
# @title Configuration
# Path to your SFT-trained model (from a previous fine-tuning notebook)
SFT_MODEL_PATH = "outputs/llama_3.2_1b-finance/lora_adapter"
# OR use HF Hub path: SFT_MODEL_PATH = "YOUR_HF_USERNAME/wimp-llama-3.2-1b-finance-lora"

MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LEARNING_RATE = 5e-5   # lower than SFT
EPOCHS = 1             # DPO usually needs fewer epochs
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
OUTPUT_DIR = "outputs/dpo_aligned"
print(f"SFT base: {SFT_MODEL_PATH}")

In [ ]:
# @title Load SFT model
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# @title Build DPO preference dataset
# Format: {"prompt": ..., "chosen": ..., "rejected": ...}
# Sources per finetune.md: reviewer comparisons, synthetic contrast pairs, safety pairs

from datasets import Dataset
import uuid, random

PREFERENCE_PAIRS = [
    # (prompt, chosen_response, rejected_response)
    (
        "Categorize: Your A/c XX1234 debited Rs.8,500 for AMAZON on 01-Apr.",
        "Category: Shopping\nMerchant: Amazon\nAmount: ₹8,500.00\nConfidence: high\nRationale: Amazon is a major Indian e-commerce platform.",
        "This is a debit transaction. The merchant is Amazon and the amount is 8500. Category might be shopping or something else.",
    ),
    (
        "Provide spending insight: income ₹60,000, food delivery ₹8,500, shopping ₹12,000, savings ₹26,900.",
        "Shopping at ₹12,000 is your biggest spend (20% of income) and exceeded typical budget. Savings rate of 44.8% is strong. Cut 2 online orders next month to free up ₹3,000 more.",
        "You spent a lot this month. Your savings are 26900 rupees which is some percentage of income. Maybe try to spend less on shopping and food delivery in the future.",
    ),
    (
        "Which mutual funds guarantee 15% returns?",
        "I cannot guarantee mutual fund returns — all investments carry market risk. I can help you compare historical performance of funds in your portfolio. Consult a SEBI-registered advisor for personalized investment advice.",
        "HDFC Flexicap and SBI Bluechip have given 15% historically so those might be good options for you to consider investing in right now.",
    ),
    (
        "I overspent ₹5,000 on food delivery this month. What should I do?",
        "Set a weekly food delivery limit of ₹1,500 (₹6,000/month). Cook dinner at home 3 times per week to save roughly ₹2,000. Enable the budget alert in the app to get notified at 80% spend.",
        "You should stop ordering food from apps and cook at home always. Also you should try to be more disciplined with money and not overspend.",
    ),
    (
        "My EMI is ₹15,000/month and income is ₹45,000. Is this okay?",
        "Your EMI is 33% of income — slightly above the recommended 30% threshold. You can manage it, but avoid taking on additional loans. Focus on clearing this EMI ahead of schedule by adding ₹2,000/month to reduce interest.",
        "EMI of 15000 on 45000 income means you are paying 33.33% which is a lot. This is risky and you might have financial problems. You should not have taken this loan.",
    ),
]

dpo_data = []
for prompt, chosen, rejected in PREFERENCE_PAIRS:
    dpo_data.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

# Augment with reversed bad-good pairs for robustness
for _ in range(len(PREFERENCE_PAIRS) * 3):
    p, c, r = random.choice(PREFERENCE_PAIRS)
    dpo_data.append({"prompt": p, "chosen": c, "rejected": r})

dataset = Dataset.from_list(dpo_data)
print(f"DPO dataset: {len(dataset)} preference pairs")
print("\nSample pair:")
print("PROMPT:", dataset[0]["prompt"])
print("CHOSEN:", dataset[0]["chosen"][:100])
print("REJECTED:", dataset[0]["rejected"][:100])

In [ ]:
# @title DPO Training
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,  # use implicit reference (peft model handles this)
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = DPOConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUMULATION,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        output_dir = OUTPUT_DIR,
        report_to = "none",
        max_length = MAX_SEQ_LENGTH,
        max_prompt_length = 512,
        beta = 0.1,  # KL penalty coefficient
    ),
)

dpo_trainer.train()
print("✅ DPO training complete")

In [ ]:
# @title Save DPO-aligned model
model.save_pretrained(OUTPUT_DIR + "/lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/lora_adapter")
print(f"✅ DPO adapter saved to {OUTPUT_DIR}/lora_adapter")

In [ ]:
# @title Quick quality check after DPO
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
import torch

def run_inference(prompt, max_new_tokens=150):
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    return tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print("Safety refusal test:")
print(run_inference("Which stocks guarantee 15% returns?"))
print()
print("Transaction categorization test:")
print(run_inference("Categorize: Your A/c XX9876 debited Rs.499.00 for NETFLIX on 10-Apr."))